# Gold models and analytical marts


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from config import (
    SILVER_NAMESPACE,
    GOLD_NAMESPACE,
    MOVIELENS_SNAPSHOT_DATE,
    IMDB_SNAPSHOT_DATE,
)

In [ ]:
from datetime import datetime, timezone
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql import types as T


spark.conf.set("spark.sql.session.timeZone", "UTC")

GOLD_BUILT_AT = datetime.now(timezone.utc).replace(tzinfo=None)


def read_snapshot(table_name, snapshot_date):
    return (
        spark.table(f"{SILVER_NAMESPACE}.{table_name}")
        .filter(F.col("_source_snapshot_date") == F.lit(snapshot_date))
    )


def add_gold_metadata(
    df,
    include_movielens=False,
    include_imdb=False,
):
    result = df

    if include_movielens:
        result = result.withColumn(
            "_movielens_snapshot_date",
            F.lit(MOVIELENS_SNAPSHOT_DATE).cast("date"),
        )

    if include_imdb:
        result = result.withColumn(
            "_imdb_snapshot_date",
            F.lit(IMDB_SNAPSHOT_DATE).cast("date"),
        )

    return result.withColumn(
        "_gold_built_at",
        F.lit(GOLD_BUILT_AT).cast("timestamp"),
    )


def safe_weighted_average(value_column, weight_column):
    weighted_sum = F.sum(F.col(value_column) * F.col(weight_column))
    total_weight = F.sum(F.col(weight_column))

    return F.when(
        total_weight > 0,
        weighted_sum / total_weight,
    )


def write_gold_table(table_name, df):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_NAMESPACE}.{table_name}")
    )


## Select source snapshots

Both dates are set in the shared `notebooks/config.py` file. Update the IMDb date there after a complete future IMDb refresh.

In [9]:
ml_movie_df = read_snapshot(
    "movielens_movie",
    MOVIELENS_SNAPSHOT_DATE,
)
ml_movie_genre_df = read_snapshot(
    "movielens_movie_genre",
    MOVIELENS_SNAPSHOT_DATE,
)
ml_link_df = read_snapshot(
    "movielens_link",
    MOVIELENS_SNAPSHOT_DATE,
)
ml_rating_df = read_snapshot(
    "movielens_rating",
    MOVIELENS_SNAPSHOT_DATE,
)
ml_tag_df = read_snapshot(
    "movielens_tag",
    MOVIELENS_SNAPSHOT_DATE,
)

imdb_name_df = read_snapshot(
    "imdb_name",
    IMDB_SNAPSHOT_DATE,
)
imdb_title_df = read_snapshot(
    "imdb_title",
    IMDB_SNAPSHOT_DATE,
)
imdb_title_genre_df = read_snapshot(
    "imdb_title_genre",
    IMDB_SNAPSHOT_DATE,
)
imdb_title_crew_member_df = read_snapshot(
    "imdb_title_crew_member",
    IMDB_SNAPSHOT_DATE,
)
imdb_title_principal_df = read_snapshot(
    "imdb_title_principal",
    IMDB_SNAPSHOT_DATE,
)
imdb_title_principal_character_df = read_snapshot(
    "imdb_title_principal_character",
    IMDB_SNAPSHOT_DATE,
)
imdb_title_rating_df = read_snapshot(
    "imdb_title_rating",
    IMDB_SNAPSHOT_DATE,
)


## Movie dimension, genre bridge, and facts


In [10]:
movie_link_df = (
    ml_movie_df.alias("movie")
    .join(
        ml_link_df.alias("link"),
        F.col("movie.movieId") == F.col("link.movieId"),
        "left",
    )
    .select(
        F.col("movie.movieId"),
        F.col("movie.title"),
        F.col("movie.releaseYear"),
        F.when(
            F.col("link.tconst").rlike(r"^tt[0-9]+$"),
            F.substring(F.col("link.tconst"), 3, 20).cast("long"),
        ).alias("imdbId"),
        F.col("link.tconst"),
        F.col("link.tmdbId"),
    )
)

dim_movie_base_df = (
    movie_link_df.alias("movie")
    .join(
        imdb_title_df.alias("imdb"),
        F.col("movie.tconst") == F.col("imdb.tconst"),
        "left",
    )
    .select(
        F.col("movie.movieId"),
        F.col("movie.imdbId"),
        F.col("movie.tconst"),
        F.col("movie.tmdbId"),
        F.col("movie.title"),
        F.col("imdb.primaryTitle"),
        F.col("imdb.originalTitle"),
        F.col("movie.releaseYear"),
        F.col("imdb.startYear"),
        F.col("imdb.endYear"),
        F.coalesce(
            F.col("imdb.startYear"),
            F.col("movie.releaseYear"),
        ).alias("analysisReleaseYear"),
        F.col("imdb.runtimeMinutes"),
        F.col("imdb.titleType"),
        F.col("imdb.isAdult"),
        F.col("imdb.tconst").isNotNull().alias("isImdbLinked"),
    )
)

dim_movie_df = add_gold_metadata(
    dim_movie_base_df,
    include_movielens=True,
    include_imdb=True,
)

movie_key_df = dim_movie_base_df.select("movieId")
title_to_movie_df = (
    dim_movie_base_df
    .where(F.col("tconst").isNotNull())
    .select("movieId", "tconst")
    .distinct()
)

movielens_genre_df = (
    ml_movie_genre_df
    .join(movie_key_df, "movieId", "inner")
    .select(
        "movieId",
        F.col("genre"),
        F.lit("movielens").alias("genreSource"),
    )
)

imdb_genre_df = (
    imdb_title_genre_df
    .join(title_to_movie_df, "tconst", "inner")
    .select(
        "movieId",
        F.col("genre"),
        F.lit("imdb").alias("genreSource"),
    )
)

bridge_movie_genres_base_df = (
    movielens_genre_df
    .unionByName(imdb_genre_df)
    .filter(F.col("genre").isNotNull() & (F.trim(F.col("genre")) != ""))
    .withColumn("genre", F.trim(F.col("genre")))
    .dropDuplicates()
)

bridge_movie_genres_df = add_gold_metadata(
    bridge_movie_genres_base_df,
    include_movielens=True,
    include_imdb=True,
)

principal_character_df = (
    imdb_title_principal_character_df
    .groupBy("tconst", "ordering")
    .agg(F.array_sort(F.collect_set("character")).alias("characters"))
)

principal_credit_df = (
    imdb_title_principal_df.alias("principal")
    .join(
        title_to_movie_df.alias("movie"),
        F.col("principal.tconst") == F.col("movie.tconst"),
        "inner",
    )
    .join(
        principal_character_df.alias("character"),
        (F.col("principal.tconst") == F.col("character.tconst"))
        & (F.col("principal.ordering") == F.col("character.ordering")),
        "left",
    )
    .select(
        F.col("movie.movieId"),
        F.col("principal.nconst"),
        F.lower(F.trim(F.col("principal.category"))).alias("creditRole"),
        F.col("principal.ordering"),
        F.col("principal.job"),
        F.coalesce(
            F.col("character.characters"),
            F.from_json(F.lit("[]"), "array<string>"),
        ).alias("characters"),
        F.array(F.lit("principal")).alias("creditSources"),
    )
)

crew_credit_df = (
    imdb_title_crew_member_df.alias("crew")
    .join(
        title_to_movie_df.alias("movie"),
        F.col("crew.tconst") == F.col("movie.tconst"),
        "inner",
    )
    .select(
        F.col("movie.movieId"),
        F.col("crew.nconst"),
        F.lower(F.trim(F.col("crew.crew_role"))).alias("creditRole"),
        F.lit(None).cast("int").alias("ordering"),
        F.lit(None).cast("string").alias("job"),
        F.from_json(F.lit("[]"), "array<string>").alias("characters"),
        F.array(F.lit("crew")).alias("creditSources"),
    )
)

fct_movie_person_credit_base_df = (
    principal_credit_df
    .unionByName(crew_credit_df)
    .filter(F.col("nconst").isNotNull() & F.col("creditRole").isNotNull() & (F.col("creditRole") != ""))
    .groupBy("movieId", "nconst", "creditRole")
    .agg(
        F.min("ordering").alias("ordering"),
        F.min("job").alias("job"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("characters")))).alias("characters"),
        F.array_sort(F.array_distinct(F.flatten(F.collect_list("creditSources")))).alias("creditSources"),
    )
    .withColumn(
        "creditRoleGroup",
        F.when(
            F.col("creditRole").isin("actor", "actress"),
            F.lit("performer"),
        ).otherwise(F.col("creditRole")),
    )
)

fct_movie_person_credit_df = add_gold_metadata(
    fct_movie_person_credit_base_df,
    include_movielens=True,
    include_imdb=True,
)

credited_people_df = (
    fct_movie_person_credit_base_df
    .select("nconst")
    .distinct()
)

movie_release_df = dim_movie_base_df.select(
    "movieId",
    "analysisReleaseYear",
)

rating_movie_age_years = (
    F.year(F.col("rating.ratedAt")) - F.col("movie.analysisReleaseYear")
)

fct_movie_rating_df = add_gold_metadata(
    ml_rating_df.alias("rating")
    .join(movie_release_df.alias("movie"), "movieId", "inner")
    .select(
        F.col("rating.userId"),
        F.col("rating.movieId"),
        F.col("rating.ratedAt"),
        F.date_format(F.to_date(F.col("rating.ratedAt")), "yyyyMMdd").cast("int").alias("ratingDateKey"),
        F.col("rating.rating"),
        F.when(rating_movie_age_years >= 0, rating_movie_age_years)
        .cast("int").alias("movieAgeAtRatingYears"),
    ),
    include_movielens=True,
)

valid_ml_tag_df = ml_tag_df.filter(
    F.col("userId").isNotNull() & F.col("taggedAt").isNotNull()
)

fct_movie_tag_df = add_gold_metadata(
    valid_ml_tag_df.alias("tag")
    .join(movie_key_df, "movieId", "inner")
    .select(
        F.col("tag.userId"),
        F.col("tag.movieId"),
        F.col("tag.tag"),
        F.col("tag.taggedAt"),
        F.date_format(F.to_date(F.col("tag.taggedAt")), "yyyyMMdd").cast("int").alias("tagDateKey"),
    ),
    include_movielens=True,
)


## Reusable movie aggregates and performance mart

In [11]:
rating_aggregate_df = (
    fct_movie_rating_df
    .groupBy("movieId")
    .agg(
        F.avg("rating").alias("movielensAverageRating"),
        F.count(F.lit(1)).cast("long").alias("ratingCount"),
    )
)

tag_aggregate_df = (
    fct_movie_tag_df
    .groupBy("movieId")
    .agg(F.count(F.lit(1)).cast("long").alias("tagCount"))
)

imdb_movie_rating_df = (
    title_to_movie_df
    .join(imdb_title_rating_df, "tconst", "left")
    .select(
        "movieId",
        F.col("averageRating").alias("imdbAverageRating"),
        F.col("numVotes").alias("imdbNumVotes"),
    )
)

mart_movie_performance_df = add_gold_metadata(
    dim_movie_base_df
    .join(rating_aggregate_df, "movieId", "left")
    .join(tag_aggregate_df, "movieId", "left")
    .join(imdb_movie_rating_df, "movieId", "left")
    .fillna({"ratingCount": 0, "tagCount": 0})
    .withColumn("displayTitle", F.coalesce("primaryTitle", "title"))
    .select(
        "movieId",
        "displayTitle",
        "analysisReleaseYear",
        "ratingCount",
        "movielensAverageRating",
        "imdbAverageRating",
        "imdbNumVotes",
        "tagCount",
    ),
    include_movielens=True,
    include_imdb=True,
)


## Genre and user marts


In [12]:
movie_genre_performance_df = (
    bridge_movie_genres_base_df
    .join(
        mart_movie_performance_df.select(
            "movieId",
            "analysisReleaseYear",
            "movielensAverageRating",
            "ratingCount",
        ),
        "movieId",
        "inner",
    )
    .filter(F.col("analysisReleaseYear").isNotNull())
)

mart_genre_trends_df = add_gold_metadata(
    movie_genre_performance_df
    .groupBy("genreSource", "genre", "analysisReleaseYear")
    .agg(
        F.countDistinct("movieId").cast("long").alias("movieCount"),
        F.sum("ratingCount").cast("long").alias("ratingCount"),
        safe_weighted_average(
            "movielensAverageRating",
            "ratingCount",
        ).alias("averageRating"),
    ),
    include_movielens=True,
    include_imdb=True,
)

mart_user_preferences_df = add_gold_metadata(
    fct_movie_rating_df
    .groupBy("userId")
    .agg(
        F.count(F.lit(1)).cast("long").alias("totalRatings"),
        F.avg("rating").alias("averageRatingGiven"),
        F.min("ratedAt").alias("firstRatingAt"),
        F.max("ratedAt").alias("lastRatingAt"),
    ),
    include_movielens=True,
    include_imdb=True,
)


## Creator mart


In [13]:
credit_movie_performance_df = (
    fct_movie_person_credit_base_df
    .join(
        mart_movie_performance_df.select(
            "movieId",
            "movielensAverageRating",
            "ratingCount",
        ),
        "movieId",
        "left",
    )
)

mart_creator_performance_df = add_gold_metadata(
    credit_movie_performance_df
    .groupBy("nconst", "creditRole")
    .agg(
        F.countDistinct("movieId").cast("long").alias("creditedTitleCount"),
        F.countDistinct(
            F.when(F.col("ratingCount") > 0, F.col("movieId"))
        ).cast("long").alias("ratedTitleCount"),
        F.avg("movielensAverageRating").alias("averageTitleRating"),
        F.sum("ratingCount").cast("long").alias("totalRatingCount"),
    )
    .join(
        imdb_name_df.select("nconst", "primaryName"),
        "nconst",
        "left",
    )
    .fillna({"totalRatingCount": 0})
    .select(
        "nconst",
        "primaryName",
        "creditRole",
        "creditedTitleCount",
        "ratedTitleCount",
        "averageTitleRating",
        "totalRatingCount",
    ),
    include_movielens=True,
    include_imdb=True,
)


## Validate

In [ ]:
gold_validation_passed = False

def duplicate_group_count(df, keys):
    return (
        df.groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )


def null_key_count(df, keys):
    condition = reduce(
        lambda left, right: left | right,
        [F.col(key).isNull() for key in keys],
    )
    return df.filter(condition).count()


quality_checks = []


def add_check(check_name, failed_row_count, is_blocking=True):
    quality_checks.append((check_name, int(failed_row_count), is_blocking))


model_grains = {
    "dim_movie": (dim_movie_df, ["movieId"]),
    "bridge_movie_genres": (
        bridge_movie_genres_df,
        ["movieId", "genre", "genreSource"],
    ),
    "fct_movie_rating": (
        fct_movie_rating_df,
        ["userId", "movieId", "ratedAt"],
    ),
    "fct_movie_tag": (
        fct_movie_tag_df,
        [
            "userId",
            "movieId",
            "tag",
            "taggedAt",
        ],
    ),
    "fct_movie_person_credit": (
        fct_movie_person_credit_df,
        ["movieId", "nconst", "creditRole"],
    ),
    "mart_movie_performance": (
        mart_movie_performance_df,
        ["movieId"],
    ),
    "mart_genre_trends": (
        mart_genre_trends_df,
        ["genreSource", "genre", "analysisReleaseYear"],
    ),
    "mart_user_preferences": (
        mart_user_preferences_df,
        ["userId"],
    ),
    "mart_creator_performance": (
        mart_creator_performance_df,
        ["nconst", "creditRole"],
    ),
}

required_keys = {
    "fct_movie_tag": ["userId", "movieId", "taggedAt"],
}

for model_name, (model_df, keys) in model_grains.items():
    add_check(
        f"{model_name}_duplicate_grain",
        duplicate_group_count(model_df, keys),
    )
    add_check(
        f"{model_name}_null_key",
        null_key_count(model_df, required_keys.get(model_name, keys)),
    )

add_check(
    "rating_fact_row_count_mismatch",
    abs(fct_movie_rating_df.count() - ml_rating_df.count()),
)
add_check(
    "tag_fact_row_count_mismatch",
    abs(fct_movie_tag_df.count() - valid_ml_tag_df.count()),
)
add_check(
    "tag_missing_user_id",
    ml_tag_df.filter(F.col("userId").isNull()).count(),
    is_blocking=False,
)
add_check(
    "tag_missing_timestamp",
    ml_tag_df.filter(F.col("taggedAt").isNull()).count(),
    is_blocking=False,
)
add_check(
    "tag_text_missing",
    fct_movie_tag_df.filter(F.col("tag").isNull()).count(),
    is_blocking=False,
)

add_check(
    "credit_person_missing_imdb_name",
    credited_people_df.join(
        imdb_name_df.select("nconst").distinct(),
        "nconst",
        "left_anti",
    ).count(),
    is_blocking=False,
)
add_check(
    "rating_invalid_value",
    fct_movie_rating_df.filter(
        (F.col("rating") < 0.5) | (F.col("rating") > 5.0) | F.col("rating").isNull()
    ).count(),
)
add_check(
    "rating_precedes_release_year",
    ml_rating_df.join(movie_release_df, "movieId", "inner")
    .filter(F.year(F.col("ratedAt")) < F.col("analysisReleaseYear"))
    .count(),
    is_blocking=False,
)

add_check(
    "movie_performance_count_mismatch",
    abs(mart_movie_performance_df.count() - dim_movie_df.count()),
)

for mart_name in (
    "mart_movie_performance",
    "mart_genre_trends",
    "mart_user_preferences",
    "mart_creator_performance",
):
    mart_df = model_grains[mart_name][0]
    add_check(
        f"{mart_name}_empty",
        1 if mart_df.limit(1).count() == 0 else 0,
    )

quality_schema = T.StructType(
    [
        T.StructField("checkName", T.StringType(), False),
        T.StructField("failedRowCount", T.LongType(), False),
        T.StructField("isBlocking", T.BooleanType(), False),
    ]
)
quality_result_df = spark.createDataFrame(
    quality_checks,
    schema=quality_schema,
)

display(quality_result_df.orderBy("checkName"))

failed_checks = [
    row.asDict()
    for row in quality_result_df.filter(
        F.col("isBlocking") & (F.col("failedRowCount") > 0)
    ).collect()
]

if failed_checks:
    raise RuntimeError(
        "Gold validation failed: "
        + ", ".join(
            f"{row['checkName']}={row['failedRowCount']}"
            for row in failed_checks
        )
    )

gold_validation_passed = True


## Persist


In [ ]:
if not globals().get("gold_validation_passed", False):
    raise RuntimeError("Run Gold validation successfully before persisting.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_NAMESPACE}")

gold_models = {
    "dim_movie": dim_movie_df,
    "bridge_movie_genres": bridge_movie_genres_df,
    "fct_movie_rating": fct_movie_rating_df,
    "fct_movie_tag": fct_movie_tag_df,
    "fct_movie_person_credit": fct_movie_person_credit_df,
    "mart_movie_performance": mart_movie_performance_df,
    "mart_genre_trends": mart_genre_trends_df,
    "mart_user_preferences": mart_user_preferences_df,
    "mart_creator_performance": mart_creator_performance_df,
}

for table_name, model_df in gold_models.items():
    write_gold_table(table_name, model_df)